<a href="https://colab.research.google.com/github/mhmddirany/Video-Transcription-Hebrew-ASR-Pipeline/blob/main/notebooks/hebrew_transcript_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers accelerate sentencepiece librosa soundfile pydub pandas openpyxl
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

WORK_DIR = "/content/facebook_test"
os.makedirs(WORK_DIR, exist_ok=True)

# Your video/audio in Drive
INPUT_VIDEO = "/content/drive/MyDrive/hebrew/hebrew.mp4"   # change this

# Your uploaded reference JSON
REF_JSON = "/content/drive/MyDrive/hebrew/hebrew_youtube_transcript/youtube_hebrew_transcript_30s_chunks_first15min.json"

AUDIO_15MIN = f"{WORK_DIR}/audio_15min.wav"
CHUNKS_DIR = f"{WORK_DIR}/chunks_30s"
RESULT_CSV = f"{WORK_DIR}/facebook_comparison.csv"
RESULT_XLSX = f"{WORK_DIR}/facebook_comparison.xlsx"

# Cell 4 — Convert first 15 min to WAV

In [ ]:
import subprocess

subprocess.run([
    "ffmpeg", "-y",
    "-ss", "00:00:00",
    "-t", str(15 * 60),
    "-i", INPUT_VIDEO,
    "-ac", "1",
    "-ar", "16000",
    AUDIO_15MIN
], check=True)

print("Saved:", AUDIO_15MIN)

Saved: /content/facebook_test/audio_15min.wav


# Cell 5 — Split into 30-second chunks

In [ ]:
from pydub import AudioSegment
import os

os.makedirs(CHUNKS_DIR, exist_ok=True)

audio = AudioSegment.from_wav(AUDIO_15MIN)
chunk_ms = 30 * 1000

chunk_files = []

for i, start_ms in enumerate(range(0, len(audio), chunk_ms)):
    end_ms = min(start_ms + chunk_ms, len(audio))
    chunk = audio[start_ms:end_ms]

    out_path = f"{CHUNKS_DIR}/chunk_{i:03d}.wav"
    chunk.export(out_path, format="wav")

    chunk_files.append({
        "chunk": i,
        "start_sec": start_ms / 1000,
        "end_sec": end_ms / 1000,
        "path": out_path
    })

print("Number of chunks:", len(chunk_files))
print(chunk_files[:3])

Number of chunks: 30
[{'chunk': 0, 'start_sec': 0.0, 'end_sec': 30.0, 'path': '/content/facebook_test/chunks_30s/chunk_000.wav'}, {'chunk': 1, 'start_sec': 30.0, 'end_sec': 60.0, 'path': '/content/facebook_test/chunks_30s/chunk_001.wav'}, {'chunk': 2, 'start_sec': 60.0, 'end_sec': 90.0, 'path': '/content/facebook_test/chunks_30s/chunk_002.wav'}]


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


# Cell 6 — Load Facebook model

In [ ]:
import torch
import librosa
from transformers import AutoProcessor, SeamlessM4Tv2Model

MODEL_ID = "facebook/seamless-m4t-v2-large"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = SeamlessM4Tv2Model.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype
).to(device)

model.eval()

print("Facebook model loaded.")

Device: cuda


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Facebook model loaded.


# Cell 7 — Facebook transcription/translation function

In [ ]:
def facebook_audio_to_text(audio_path, tgt_lang):
    """
    tgt_lang:
    heb = Hebrew transcription
    arb = Arabic translation
    """

    audio_array, sr = librosa.load(audio_path, sr=16000, mono=True)

    inputs = processor(
        audio=audio_array,
        sampling_rate=16000,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            tgt_lang=tgt_lang,
            generate_speech=False
        )

    # Robust decoding
    if hasattr(output_tokens, "sequences"):
        token_ids = output_tokens.sequences
    elif isinstance(output_tokens, tuple):
        token_ids = output_tokens[0]
    else:
        token_ids = output_tokens

    token_ids = token_ids.detach().cpu()

    while token_ids.ndim > 1:
        token_ids = token_ids[0]

    text = processor.decode(
        token_ids.tolist(),
        skip_special_tokens=True
    )

    return text

In [ ]:
test_path = chunk_files[0]["path"]

print("Hebrew:")
print(facebook_audio_to_text(test_path, tgt_lang="heb"))

print("Arabic:")
print(facebook_audio_to_text(test_path, tgt_lang="arb"))

Hebrew:
רשת עושים היסטוריה
Arabic:
نعمل تاريخ


# Cell 8 — Run Facebook on all 30s chunks

In [ ]:
import pandas as pd
from tqdm import tqdm

rows = []

for item in tqdm(chunk_files):
    path = item["path"]

    try:
        fb_hebrew = facebook_audio_to_text(path, tgt_lang="heb")
    except Exception as e:
        fb_hebrew = "ERROR: " + str(e)

    try:
        fb_arabic = facebook_audio_to_text(path, tgt_lang="arb")
    except Exception as e:
        fb_arabic = "ERROR: " + str(e)

    rows.append({
        "chunk_id": item["chunk"] + 1,
        "start_sec": item["start_sec"],
        "end_sec": item["end_sec"],
        "facebook_hebrew": fb_hebrew,
        "facebook_arabic": fb_arabic
    })

df_fb = pd.DataFrame(rows)
df_fb.head()

100%|██████████| 30/30 [02:47<00:00,  5.58s/it]


,chunk_id,start_sec,end_sec,facebook_hebrew,facebook_arabic
0,1,0.0,30.0,רשת עושים היסטוריה,نعمل تاريخ
1,2,30.0,60.0,"אקשן לעסקים קטנים עם אגר פרץ דיין שלום, שלום, ...",و في هذا الفصل نستفيد من الناحية النظرية، في ا...
2,3,60.0,90.0,"אז באופן כללי, לפני שאנחנו עושים חלוקת בין עסק...",من ناحية ماكرون، من ناحية ماكرون، من ناحية ماك...
3,4,90.0,120.0,"על הגדרת המוצר, על מדידת המוצר, על ההגדרות החש...",بعد ذلك قمنا بتحديد المنتج وقياسه وميزاته المح...
4,5,120.0,150.0,"זה זהות חשבונאית, וזה מה זה חיסכון, זה מאוד חש...",هذا هو هوية حسابية، فماذا يعني هذا؟


## Cell 9 — Load Hebrew + Arabic reference JSON

Use the paths where you uploaded the files.

In [ ]:
import json
import pandas as pd

REF_HEBREW_JSON = "/content/drive/MyDrive/hebrew/hebrew_youtube_transcript/youtube_hebrew_transcript_30s_chunks_first15min.json"

REF_ARABIC_JSON = "/content/drive/MyDrive/hebrew/hebrew_youtube_transcript/youtube_hebrew_transcript_30s_chunks_first15min_arabic.json"

with open(REF_HEBREW_JSON, "r", encoding="utf-8") as f:
    hebrew_data = json.load(f)

with open(REF_ARABIC_JSON, "r", encoding="utf-8") as f:
    arabic_data = json.load(f)

df_ref_he = pd.DataFrame(hebrew_data["chunks"])
df_ref_ar = pd.DataFrame(arabic_data["chunks"])

df_ref_he.head()

,chunk_id,start,start_sec,end,end_sec,youtube_hebrew
0,1,00:00:00,0,00:00:30,30,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עו...
1,2,00:00:30,30,00:01:00,60,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשר...
2,3,00:01:00,60,00:01:30,90,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הל...
3,4,00:01:30,90,00:02:00,120,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה...
4,5,00:02:00,120,00:02:30,150,החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה ...


In [ ]:
REF_HEBREW_JSON = "/content/drive/MyDrive/hebrew/youtube_hebrew_transcript_30s_chunks_first15min.json"
REF_ARABIC_JSON = "/content/drive/MyDrive/hebrew/youtube_hebrew_transcript_30s_chunks_first15min_arabic.json"

# Cell 10 — Prepare reference table

In [ ]:
# Hebrew reference column can be called youtube_hebrew or text
if "youtube_hebrew" in df_ref_he.columns:
    df_ref_he["youtube_hebrew_reference"] = df_ref_he["youtube_hebrew"]
elif "text" in df_ref_he.columns:
    df_ref_he["youtube_hebrew_reference"] = df_ref_he["text"]
else:
    raise ValueError("Hebrew text column not found.")

# Arabic reference column can be called text_arabic or arabic_translation
if "text_arabic" in df_ref_ar.columns:
    df_ref_ar["reference_arabic_translation"] = df_ref_ar["text_arabic"]
elif "arabic_translation" in df_ref_ar.columns:
    df_ref_ar["reference_arabic_translation"] = df_ref_ar["arabic_translation"]
else:
    raise ValueError("Arabic text column not found.")

df_ref = df_ref_he[[
    "chunk_id",
    "start_sec",
    "end_sec",
    "youtube_hebrew_reference"
]].copy()

df_ref["reference_arabic_translation"] = df_ref_ar["reference_arabic_translation"]

df_ref.head()

,chunk_id,start_sec,end_sec,youtube_hebrew_reference,reference_arabic_translation
0,1,0,30,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עו...,شبكة 'نصنع التاريخ'. شبكة 'نصنع التاريخ'. برنا...
1,2,30,60,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשר...,أهلاً شيرا، أهلاً عومر، أهلاً هاجر. في هذه الح...
2,3,60,90,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הל...,هل تحرّك هذه الأعمال شيئاً؟ بشكل عام، قبل تقسي...
3,4,90,120,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה...,في سلسلة قياس الناتج تحدثنا عن تعريف الناتج وق...
4,5,120,150,החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה ...,الادخار يساوي الاستثمار تماماً، وهذه هوية محاس...


# Cell 11 — Merge reference + Facebook results

In [ ]:
df_compare = pd.merge(
    df_ref,
    df_fb[["chunk_id", "facebook_hebrew", "facebook_arabic"]],
    on="chunk_id",
    how="left"
)

df_compare["manual_score_facebook_hebrew"] = ""
df_compare["manual_score_facebook_arabic"] = ""
df_compare["notes"] = ""

df_compare.head()

,chunk_id,start_sec,end_sec,youtube_hebrew_reference,reference_arabic_translation,facebook_hebrew,facebook_arabic,manual_score_facebook_hebrew,manual_score_facebook_arabic,notes
0,1,0,30,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עו...,شبكة 'نصنع التاريخ'. شبكة 'نصنع التاريخ'. برنا...,רשת עושים היסטוריה,نعمل تاريخ,,,
1,2,30,60,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשר...,أهلاً شيرا، أهلاً عومر، أهلاً هاجر. في هذه الح...,"אקשן לעסקים קטנים עם אגר פרץ דיין שלום, שלום, ...",و في هذا الفصل نستفيد من الناحية النظرية، في ا...,,,
2,3,60,90,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הל...,هل تحرّك هذه الأعمال شيئاً؟ بشكل عام، قبل تقسي...,"אז באופן כללי, לפני שאנחנו עושים חלוקת בין עסק...",من ناحية ماكرون، من ناحية ماكرون، من ناحية ماك...,,,
3,4,90,120,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה...,في سلسلة قياس الناتج تحدثنا عن تعريف الناتج وق...,"על הגדרת המוצר, על מדידת המוצר, על ההגדרות החש...",بعد ذلك قمنا بتحديد المنتج وقياسه وميزاته المح...,,,
4,5,120,150,החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה ...,الادخار يساوي الاستثمار تماماً، وهذه هوية محاس...,"זה זהות חשבונאית, וזה מה זה חיסכון, זה מאוד חש...",هذا هو هوية حسابية، فماذا يعني هذا؟,,,


In [ ]:
RESULT_CSV = "/content/facebook_test/facebook_comparison_first15min.csv"
RESULT_XLSX = "/content/facebook_test/facebook_comparison_first15min.xlsx"

df_compare.to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")
df_compare.to_excel(RESULT_XLSX, index=False)

print("Saved CSV:", RESULT_CSV)
print("Saved Excel:", RESULT_XLSX)

Saved CSV: /content/facebook_test/facebook_comparison_first15min.csv
Saved Excel: /content/facebook_test/facebook_comparison_first15min.xlsx


# Cell 13 — Copy results to Drive

In [ ]:
import os, shutil

DRIVE_OUT = "/content/drive/MyDrive/hebrew/facebook_results"
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy(RESULT_CSV, DRIVE_OUT)
shutil.copy(RESULT_XLSX, DRIVE_OUT)

print("Copied results to:", DRIVE_OUT)

Copied results to: /content/drive/MyDrive/hebrew/facebook_results


# Cell 14 — Install WER/CER library

In [ ]:
!pip -q install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.8 MB/s eta 0:00:00


# Cell 15 — Compute WER/CER for Hebrew transcription

In [ ]:
import re
import pandas as pd
from jiwer import wer, cer

def clean_hebrew_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # remove Hebrew vowels / niqqud
    text = re.sub(r"[\u0591-\u05C7]", "", text)

    # remove punctuation/symbols
    text = re.sub(r"[^\u0590-\u05FF\s]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df_eval_he = df_compare.copy()

df_eval_he["ref_clean"] = df_eval_he["youtube_hebrew_reference"].apply(clean_hebrew_text)
df_eval_he["hyp_clean"] = df_eval_he["facebook_hebrew"].apply(clean_hebrew_text)

# remove empty refs and ERROR rows
df_eval_he = df_eval_he[
    (df_eval_he["ref_clean"] != "") &
    (~df_eval_he["facebook_hebrew"].astype(str).str.startswith("ERROR"))
].copy()

refs = df_eval_he["ref_clean"].tolist()
hyps = df_eval_he["hyp_clean"].tolist()

hebrew_wer = wer(refs, hyps)
hebrew_cer = cer(refs, hyps)

print("===== FACEBOOK HEBREW TRANSCRIPTION METRICS =====")
print("Number of chunks:", len(df_eval_he))
print("WER:", hebrew_wer)
print("CER:", hebrew_cer)

===== FACEBOOK HEBREW TRANSCRIPTION METRICS =====
Number of chunks: 30
WER: 0.6637813211845103
CER: 0.5551184622261958


# Cell 16 — Per-chunk WER/CER

In [ ]:
chunk_scores = []

for _, row in df_eval_he.iterrows():
    ref = row["ref_clean"]
    hyp = row["hyp_clean"]

    chunk_scores.append({
        "chunk_id": row["chunk_id"],
        "start_sec": row["start_sec"],
        "end_sec": row["end_sec"],
        "wer": wer(ref, hyp),
        "cer": cer(ref, hyp),
        "reference": row["youtube_hebrew_reference"],
        "facebook_hebrew": row["facebook_hebrew"]
    })

df_chunk_scores = pd.DataFrame(chunk_scores)

df_chunk_scores.head()

,chunk_id,start_sec,end_sec,wer,cer,reference,facebook_hebrew
0,1,0,30,0.914286,0.901639,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עו...,רשת עושים היסטוריה
1,2,30,60,0.898876,0.889831,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשר...,"אקשן לעסקים קטנים עם אגר פרץ דיין שלום, שלום, ..."
2,3,60,90,0.597015,0.491713,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הל...,"אז באופן כללי, לפני שאנחנו עושים חלוקת בין עסק..."
3,4,90,120,0.609375,0.439655,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה...,"על הגדרת המוצר, על מדידת המוצר, על ההגדרות החש..."
4,5,120,150,0.565217,0.426934,החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה ...,"זה זהות חשבונאית, וזה מה זה חיסכון, זה מאוד חש..."


# Cell 17 — Save WER/CER results

In [ ]:
WER_CER_CSV = "/content/facebook_test/facebook_hebrew_wer_cer_chunks.csv"
WER_CER_XLSX = "/content/facebook_test/facebook_hebrew_wer_cer_chunks.xlsx"

df_chunk_scores.to_csv(WER_CER_CSV, index=False, encoding="utf-8-sig")
df_chunk_scores.to_excel(WER_CER_XLSX, index=False)

print("Saved:", WER_CER_CSV)
print("Saved:", WER_CER_XLSX)

Saved: /content/facebook_test/facebook_hebrew_wer_cer_chunks.csv
Saved: /content/facebook_test/facebook_hebrew_wer_cer_chunks.xlsx


Optional — Arabic WER/CER

In [ ]:
def clean_arabic_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # remove Arabic diacritics and tatweel
    text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)

    # keep Arabic letters and spaces only
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df_eval_ar = df_compare.copy()

df_eval_ar["ref_ar_clean"] = df_eval_ar["reference_arabic_translation"].apply(clean_arabic_text)
df_eval_ar["hyp_ar_clean"] = df_eval_ar["facebook_arabic"].apply(clean_arabic_text)

df_eval_ar = df_eval_ar[
    (df_eval_ar["ref_ar_clean"] != "") &
    (~df_eval_ar["facebook_arabic"].astype(str).str.startswith("ERROR"))
].copy()

arabic_refs = df_eval_ar["ref_ar_clean"].tolist()
arabic_hyps = df_eval_ar["hyp_ar_clean"].tolist()

arabic_wer = wer(arabic_refs, arabic_hyps)
arabic_cer = cer(arabic_refs, arabic_hyps)

print("===== FACEBOOK HEBREW→ARABIC TRANSLATION METRICS =====")
print("Number of chunks:", len(df_eval_ar))
print("Arabic WER:", arabic_wer)
print("Arabic CER:", arabic_cer)

===== FACEBOOK HEBREW→ARABIC TRANSLATION METRICS =====
Number of chunks: 30
Arabic WER: 0.9316394434361767
Arabic CER: 0.7169051404345522


# Do this now: calculate full 15-min WER, not per-chunk WER

In [ ]:
# Sort by chunk_id
df_eval_he_sorted = df_eval_he.sort_values("chunk_id")

full_ref = " ".join(df_eval_he_sorted["ref_clean"].tolist())
full_hyp = " ".join(df_eval_he_sorted["hyp_clean"].tolist())

full_wer = wer(full_ref, full_hyp)
full_cer = cer(full_ref, full_hyp)

print("===== FULL 15-MIN FACEBOOK HEBREW METRICS =====")
print("WER:", full_wer)
print("CER:", full_cer)

===== FULL 15-MIN FACEBOOK HEBREW METRICS =====
WER: 0.6542141230068337
CER: 0.5408418048867487


Facebook SeamlessM4T v2 large was tested on a 15-minute Hebrew podcast sample.
The model achieved WER = 0.654 and CER = 0.541 for Hebrew transcription.
This is a high error rate, meaning the model is not reliable enough for Hebrew-to-Hebrew transcription on this audio.
The likely reasons are fast podcast speech, multiple speakers, possible noisy YouTube captions, and the fact that SeamlessM4T is mainly a multilingual speech translation model, not a Hebrew-specialized ASR model.

In [ ]:
pd.set_option("display.max_colwidth", 500)

df_compare[[
    "chunk_id",
    "youtube_hebrew_reference",
    "reference_arabic_translation",
    "facebook_hebrew",
    "facebook_arabic"
]].head(10)

,chunk_id,youtube_hebrew_reference,reference_arabic_translation,facebook_hebrew,facebook_arabic
0,1,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עושים היסטוריה. עושים חשבון עם פרופסור עומר מואב ושירה הדס נקר. אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשראי לעסקים קטנים עם אגר פרץ דיין שלום,شبكة 'نصنع التاريخ'. شبكة 'نصنع التاريخ'. برنامج 'نحسب' مع البروفيسور عومر موآف وشيرا هداس نكر. أهلاً شيرا، نحن الآن في الحلقة الثانية عن الائتمان للأعمال الصغيرة مع هاجر بيرتس ديان، مرحباً.,רשת עושים היסטוריה,نعمل تاريخ
1,2,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשראי לעסקים קטנים עם אגר פרץ דיין שלום שלום שירה שלום עומר היי אגר ובפרק הזה אנחנו נצלול יותר לזווית התיאורטית בעצם איזה מחקרים מגבים את כל הנושא שדיברנו עליו בפרק הקודם שאני ממש ממליצה להאזין לו לפני שצוללים לפרק הזה של אשראי לעסקים ובוא נפתח רגע עם הנושא המקרוכלי כי אנחנו דיברנו עד עכשיו על החברה הישראלית וכמה חשוב לנו כחברה שיהיו עסקים קטנים ולהשקיע בהם ולהוריד להם רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הלהם מזיזים משהו,أهلاً شيرا، أهلاً عومر، أهلاً هاجر. في هذه الحلقة سندخل أكثر إلى الزاوية النظرية: أي أبحاث تدعم الموضوع الذي تحدثنا عنه في الحلقة السابقة، والتي أنصح فعلاً بسماعها قبل هذه الحلقة عن ائتمان الأعمال. ولنبدأ بموضوع الاقتصاد الكلي: تحدثنا حتى الآن عن المجتمع الإسرائيلي ومدى أهمية وجود أعمال صغيرة والاستثمار فيها وتقليل التنظيم عليها، لكن هل هذا مهم فعلاً من ناحية الماكرو؟ هل تحرّك هذه الأعمال شيئاً؟,"אקשן לעסקים קטנים עם אגר פרץ דיין שלום, שלום, שלום, שלום.",و في هذا الفصل نستفيد من الناحية النظرية، في الواقع، أنا أوصي بإستماع كل ما تحدثنا عنه في هذا الفصل، و دعونا نبدأ بالمسألة الاقتصادية، لأننا تحدثنا عن أهمية الشركات الإسرائيلية كشركة صغيرة، وأن نخفض فيها
2,3,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הלהם מזיזים משהו אז אז באופן כללי לפני שעושים את החלוקה אפילו בין עסקים קטנים לגדולים צריך לדבר על החשיבות של הטיווח הפיננסי של שוק האשראי ואני רוצה להזכיר למאזינות ולמאזינים או שזה היום כבר צופות וצופים לחלק מהפודקאסטים שלנו אה שעשינו פרק למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה בכלל תוצר ההגדרה א ומדידת התוצר,هل تحرّك هذه الأعمال شيئاً؟ بشكل عام، قبل تقسيم الأعمال إلى صغيرة وكبيرة، يجب الحديث عن أهمية الوساطة المالية وسوق الائتمان. وأريد أن أذكّر المستمعات والمستمعين، أو المشاهدات والمشاهدين اليوم في بعض بودكاستاتنا، أننا قدمنا حلقة، بل سلسلة حلقات، عن قياس الناتج: ما هو الناتج أصلاً، تعريفه وقياسه.,"אז באופן כללי, לפני שאנחנו עושים חלוקת בין עסקים קטנים לגדולים, אנחנו צריכים לדבר על החשיבות של התווך הפיננסי של שוק האשראי, ואני רוצה להזכיר למאזנים, או כבר צופים, חלק מהפודקאסט שלנו, שעשינו סדרה של פרקים על מדידת המוצר.",من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون.
3,4,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה בכלל תוצר ההגדרה א ומדידת התוצר והזהויות החשבונאיות ואחר כך עשינו גם פרק על תיאוריה שמסבירה את התוצר בגדול בפרק של מדידת התוצר ראינו שיש קשר הדוק בין חיסכון להשקעה כלומר ברמה המקרוכלית אם אנחנו נתעלם מהפער בין היצוב והיוון נניח שהוא אפס אז ברמה המקרוכלית החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה זה,في سلسلة قياس الناتج تحدثنا عن تعريف الناتج وقياسه والهويات المحاسبية، ثم قدمنا حلقة عن النظرية التي تشرح الناتج بصورة عامة. في حلقة قياس الناتج رأينا أن هناك علاقة وثيقة بين الادخار والاستثمار؛ أي على مستوى الاقتصاد الكلي، إذا تجاهلنا الفرق بين التصدير والاستيراد وافترضنا أنه صفر، فإن الادخار يساوي الاستثمار تماماً. هذه هوية محاسبية. وما هو الادخار؟,"על הגדרת המוצר, על מדידת המוצר, על ההגדרות החשבונאיות, ואז גם על תיאוריה של המוצר, ראינו שיש קשר הדוק בין חיסכון להשקעה, כלומר, ברמה המקרוכלכלית, אם נניח שהפער בין היצוא ליבוא הוא אפס, אז ברמה המקרוכלית, החסכון שווה בדיוק להשקעה.",بعد ذلك قمنا بتحديد المنتج وقياسه وميزاته المحاسبية، ثم قمنا بتفسير نظرية المنتج، في الجزء الأكبر من المقياس رأينا أن هناك علاقة وثيقة بين الاقتصاد الاقتصادي والاستثمار، فإذا تجاهلنا الفجوة بين الصادرات والاستيراد، فإذا كان الاقتصاد الاقتصادي هو بالضبط ما يستحقه الاستثمار.
4,5,"החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה זה חיסכון? בואי נעשה פה סדר כי זה מאוד חשוב להבחין בין חיסכון להשקעה, כי ב

# Cell 14 — Install / import Whisper model

In [ ]:
!pip -q install -U transformers accelerate librosa soundfile jiwer

# Cell 15 — Load Whisper large-v3

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WHISPER_MODEL_ID = "openai/whisper-large-v3"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)

whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL_ID)

whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True
).to(device)

whisper_asr = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=0 if device == "cuda" else -1
)

print("Whisper loaded.")

Device: cuda


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Whisper loaded.


# Cell 16 — Test one chunk first

In [ ]:
test_path = chunk_files[0]["path"]

out = whisper_asr(
    test_path,
    generate_kwargs={
        "language": "he",
        "task": "transcribe"
    }
)

print(out["text"])

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


 רשת עושים היסטוריה עושים חשבון עם פרופסור עומר מואב ושירה הדס נקר


# Cell 17 — Run Whisper on all chunks

In [ ]:
import pandas as pd
from tqdm import tqdm

rows = []

for item in tqdm(chunk_files):
    path = item["path"]

    try:
        out = whisper_asr(
            path,
            generate_kwargs={
                "language": "he",
                "task": "transcribe"
            }
        )
        whisper_hebrew = out["text"]

    except Exception as e:
        whisper_hebrew = "ERROR: " + str(e)

    rows.append({
        "chunk_id": item["chunk"] + 1,
        "start_sec": item["start_sec"],
        "end_sec": item["end_sec"],
        "whisper_hebrew": whisper_hebrew
    })

df_whisper = pd.DataFrame(rows)
df_whisper.head()

100%|██████████| 30/30 [04:04<00:00,  8.16s/it]


,chunk_id,start_sec,end_sec,whisper_hebrew
0,1,0.0,30.0,רשת עושים היסטוריה עושים חשבון עם פרופסור עומר מואב ושירה הדס נקר
1,2,30.0,60.0,"אשראי לעסקים קטנים עם אגר פרץ דיין, שלום. שלום שירה, שלום עומר. היי אגר. ובפרק הזה אנחנו נצלול יותר לזווית התיאורטית, בעצם איזה מחקרים מגבים את כל הנושא שדיברנו עליו בפרק הקודם, שאני ממש ממליצה להאזין לו לפני שצוללים לפרק הזה, של אשראי לעסקים, ובואו נפתח רגע עם הנושא המקרו-כלכלי, כי אנחנו דיברנו עוד עכשיו על החברה הישראלית, וכמה חשוב לנו כחברה שיהיו עסקים קטנים, ולהשקיע בהם ולהוריד להם רגולציה,"
2,3,60.0,90.0,"חשוב בכלל מבחינת מקרו? העסקים האלה מזיזים משהו? אז באופן כללי, לפני שעושים את החלוקה אפילו בין עסקים קטנים לגדולים, צריך לדבר על החשיבות של הטיבוך הפיננסי של שוק האשראי. ואני רוצה להזכיר למאזינות ולמאזינים, או שזה היום כבר צופות וצופים בחלק מהפודקאסטים שלנו, שעשינו פרק למעשה סדרה של פרקים על מדידת התוצר."
3,4,90.0,120.0,"את התוצר, הגדרה ומדידת התוצר והזהויות החשבונאיות. ורק עשינו גם פרק על תיאוריה שמסבירה את התוצר. בגדול, בפרק של מדידת התוצר ראינו שיש קשר הדוק בין חיסכון להשקעה. כלומר, ברמה המאקרו-כלכלית, אם אנחנו נתעלם מהפער בין הייצוב והייבון, נניח שהוא אפס, אז ברמה המאקרו-כלכלית, החיסכון שווה בדיוק להשקעה."
4,5,120.0,150.0,זו זהות חשבונאית ומה זה חיסכון בואי נעשה פה סדר כי זה מאוד חשוב להבחין בין חיסכון להשקע כי ביום יום אנשים מבלבלים בין שני המושגים האלו. חיסכון זה זרם זאת אומרת ההפרש בין הכנסה להוצאה למשל החיסכון השנתי של משק הבית של האכשירה זה ההפרש בין ההכנסה השנתית שיש לכם לבין ההוצאה השנתית את ההפרש חסכתם.


# Cell 18 — Merge Whisper with your comparison table

In [ ]:
df_compare2 = pd.merge(
    df_compare,
    df_whisper[["chunk_id", "whisper_hebrew"]],
    on="chunk_id",
    how="left"
)

df_compare2.head()

,chunk_id,start_sec,end_sec,youtube_hebrew_reference,reference_arabic_translation,facebook_hebrew,facebook_arabic,manual_score_facebook_hebrew,manual_score_facebook_arabic,notes,whisper_hebrew
0,1,0,30,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עושים היסטוריה. עושים חשבון עם פרופסור עומר מואב ושירה הדס נקר. אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשראי לעסקים קטנים עם אגר פרץ דיין שלום,شبكة 'نصنع التاريخ'. شبكة 'نصنع التاريخ'. برنامج 'نحسب' مع البروفيسور عومر موآف وشيرا هداس نكر. أهلاً شيرا، نحن الآن في الحلقة الثانية عن الائتمان للأعمال الصغيرة مع هاجر بيرتس ديان، مرحباً.,רשת עושים היסטוריה,نعمل تاريخ,,,,רשת עושים היסטוריה עושים חשבון עם פרופסור עומר מואב ושירה הדס נקר
1,2,30,60,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשראי לעסקים קטנים עם אגר פרץ דיין שלום שלום שירה שלום עומר היי אגר ובפרק הזה אנחנו נצלול יותר לזווית התיאורטית בעצם איזה מחקרים מגבים את כל הנושא שדיברנו עליו בפרק הקודם שאני ממש ממליצה להאזין לו לפני שצוללים לפרק הזה של אשראי לעסקים ובוא נפתח רגע עם הנושא המקרוכלי כי אנחנו דיברנו עד עכשיו על החברה הישראלית וכמה חשוב לנו כחברה שיהיו עסקים קטנים ולהשקיע בהם ולהוריד להם רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הלהם מזיזים משהו,أهلاً شيرا، أهلاً عومر، أهلاً هاجر. في هذه الحلقة سندخل أكثر إلى الزاوية النظرية: أي أبحاث تدعم الموضوع الذي تحدثنا عنه في الحلقة السابقة، والتي أنصح فعلاً بسماعها قبل هذه الحلقة عن ائتمان الأعمال. ولنبدأ بموضوع الاقتصاد الكلي: تحدثنا حتى الآن عن المجتمع الإسرائيلي ومدى أهمية وجود أعمال صغيرة والاستثمار فيها وتقليل التنظيم عليها، لكن هل هذا مهم فعلاً من ناحية الماكرو؟ هل تحرّك هذه الأعمال شيئاً؟,"אקשן לעסקים קטנים עם אגר פרץ דיין שלום, שלום, שלום, שלום.",و في هذا الفصل نستفيد من الناحية النظرية، في الواقع، أنا أوصي بإستماع كل ما تحدثنا عنه في هذا الفصل، و دعونا نبدأ بالمسألة الاقتصادية، لأننا تحدثنا عن أهمية الشركات الإسرائيلية كشركة صغيرة، وأن نخفض فيها,,,,"אשראי לעסקים קטנים עם אגר פרץ דיין, שלום. שלום שירה, שלום עומר. היי אגר. ובפרק הזה אנחנו נצלול יותר לזווית התיאורטית, בעצם איזה מחקרים מגבים את כל הנושא שדיברנו עליו בפרק הקודם, שאני ממש ממליצה להאזין לו לפני שצוללים לפרק הזה, של אשראי לעסקים, ובואו נפתח רגע עם הנושא המקרו-כלכלי, כי אנחנו דיברנו עוד עכשיו על החברה הישראלית, וכמה חשוב לנו כחברה שיהיו עסקים קטנים, ולהשקיע בהם ולהוריד להם רגולציה,"
2,3,60,90,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הלהם מזיזים משהו אז אז באופן כללי לפני שעושים את החלוקה אפילו בין עסקים קטנים לגדולים צריך לדבר על החשיבות של הטיווח הפיננסי של שוק האשראי ואני רוצה להזכיר למאזינות ולמאזינים או שזה היום כבר צופות וצופים לחלק מהפודקאסטים שלנו אה שעשינו פרק למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה בכלל תוצר ההגדרה א ומדידת התוצר,هل تحرّك هذه الأعمال شيئاً؟ بشكل عام، قبل تقسيم الأعمال إلى صغيرة وكبيرة، يجب الحديث عن أهمية الوساطة المالية وسوق الائتمان. وأريد أن أذكّر المستمعات والمستمعين، أو المشاهدات والمشاهدين اليوم في بعض بودكاستاتنا، أننا قدمنا حلقة، بل سلسلة حلقات، عن قياس الناتج: ما هو الناتج أصلاً، تعريفه وقياسه.,"אז באופן כללי, לפני שאנחנו עושים חלוקת בין עסקים קטנים לגדולים, אנחנו צריכים לדבר על החשיבות של התווך הפיננסי של שוק האשראי, ואני רוצה להזכיר למאזנים, או כבר צופים, חלק מהפודקאסט שלנו, שעשינו סדרה של פרקים על מדידת המוצר.",من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون، من ناحية ماكرون.,,,,"חשוב בכלל מבחינת מקרו? העסקים האלה מזיזים משהו? אז באופן כללי, לפני שעושים את החלוקה אפילו בין עסקים קטנים לגדולים, צריך לדבר על החשיבות של הטיבוך הפיננסי של שוק האשראי. ואני רוצה להזכיר למאזינות ולמאזינים, או שזה היום כבר צופות וצופים בחלק מהפודקאסטים שלנו, שעשינו פרק למעשה סדרה של פרקים על מדידת התוצר."
3,4,90,120,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה בכלל תוצר ההגדרה א ומדידת התוצר והזהויות החשבונאיות ואחר כך עשינו גם פרק על תיאוריה שמסבירה את התוצר בגדול בפרק של מדידת התוצר ראינו שיש קשר הדוק בין חיסכון להשקעה כלומר ברמה המקרוכלית אם אנחנו נתעלם מהפער בין היצוב והיוון נניח שהוא אפס אז ברמה המקרוכלית החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה זה,في سلسلة قياس الناتج تحدثنا عن تعريف الناتج وقياسه والهويات المحاسبية، ثم قدمنا 

Cell 19 — Compute Whisper WER/CER

In [ ]:
import re
from jiwer import wer, cer
import pandas as pd

def clean_hebrew_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove Hebrew vowels / niqqud
    text = re.sub(r"[\u0591-\u05C7]", "", text)

    # Keep only Hebrew letters and spaces
    text = re.sub(r"[^\u0590-\u05FF\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df_eval_whisper = df_compare2.copy()

df_eval_whisper["ref_clean"] = df_eval_whisper["youtube_hebrew_reference"].apply(clean_hebrew_text)
df_eval_whisper["hyp_clean"] = df_eval_whisper["whisper_hebrew"].apply(clean_hebrew_text)

df_eval_whisper = df_eval_whisper[
    (df_eval_whisper["ref_clean"] != "") &
    (~df_eval_whisper["whisper_hebrew"].astype(str).str.startswith("ERROR"))
].copy()

refs = df_eval_whisper["ref_clean"].tolist()
hyps = df_eval_whisper["hyp_clean"].tolist()

whisper_wer = wer(refs, hyps)
whisper_cer = cer(refs, hyps)

print("===== WHISPER HEBREW TRANSCRIPTION METRICS =====")
print("Number of chunks:", len(df_eval_whisper))
print("WER:", whisper_wer)
print("CER:", whisper_cer)

===== WHISPER HEBREW TRANSCRIPTION METRICS =====
Number of chunks: 30
WER: 0.27517084282460136
CER: 0.2264640143048726


# Cell 20 — Full 15-min WER/CER

In [ ]:
df_eval_whisper_sorted = df_eval_whisper.sort_values("chunk_id")

full_ref = " ".join(df_eval_whisper_sorted["ref_clean"].tolist())
full_hyp = " ".join(df_eval_whisper_sorted["hyp_clean"].tolist())

full_whisper_wer = wer(full_ref, full_hyp)
full_whisper_cer = cer(full_ref, full_hyp)

print("===== FULL 15-MIN WHISPER METRICS =====")
print("WER:", full_whisper_wer)
print("CER:", full_whisper_cer)

===== FULL 15-MIN WHISPER METRICS =====
WER: 0.2715261958997722
CER: 0.21981451756732656


# ell 21 — Compare Facebook vs Whisper

In [ ]:
results_summary = pd.DataFrame([
    {
        "model": "Facebook SeamlessM4T v2 large",
        "task": "Hebrew ASR",
        "WER": 0.6542141230068337,
        "CER": 0.5408418048867487
    },
    {
        "model": "Whisper large-v3",
        "task": "Hebrew ASR",
        "WER": full_whisper_wer,
        "CER": full_whisper_cer
    }
])

results_summary

,model,task,WER,CER
0,Facebook SeamlessM4T v2 large,Hebrew ASR,0.654214,0.540842
1,Whisper large-v3,Hebrew ASR,0.271526,0.219815


# Cell 22 — Save final comparison

In [ ]:
RESULT_COMPARE_CSV = "/content/facebook_test/facebook_vs_whisper_first15min.csv"
RESULT_COMPARE_XLSX = "/content/facebook_test/facebook_vs_whisper_first15min.xlsx"

df_compare2.to_csv(RESULT_COMPARE_CSV, index=False, encoding="utf-8-sig")
df_compare2.to_excel(RESULT_COMPARE_XLSX, index=False)

SUMMARY_CSV = "/content/facebook_test/model_summary_first15min.csv"
SUMMARY_XLSX = "/content/facebook_test/model_summary_first15min.xlsx"

results_summary.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
results_summary.to_excel(SUMMARY_XLSX, index=False)

print("Saved:")
print(RESULT_COMPARE_CSV)
print(RESULT_COMPARE_XLSX)
print(SUMMARY_CSV)
print(SUMMARY_XLSX)

Saved:
/content/facebook_test/facebook_vs_whisper_first15min.csv
/content/facebook_test/facebook_vs_whisper_first15min.xlsx
/content/facebook_test/model_summary_first15min.csv
/content/facebook_test/model_summary_first15min.xlsx


# Cell 23 — Copy to Drive

In [ ]:
import os, shutil

DRIVE_OUT = "/content/drive/MyDrive/hebrew/model_comparison_results"
os.makedirs(DRIVE_OUT, exist_ok=True)

for path in [
    RESULT_COMPARE_CSV,
    RESULT_COMPARE_XLSX,
    SUMMARY_CSV,
    SUMMARY_XLSX
]:
    shutil.copy(path, DRIVE_OUT)

print("Copied to:", DRIVE_OUT)

Copied to: /content/drive/MyDrive/hebrew/model_comparison_results


1. Caspi-1.7B      ← Hebrew-specialized, best candidate
2. Ivrit.ai tuned Whisper
3. Qwen3-ASR base  ← maybe not ideal because official Qwen language list does not clearly include Hebrew

# A) Test Caspi-1.7B



Cell 24 — Install Qwen ASR

In [ ]:
!pip -q install -U qwen-asr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 140.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 73.5 MB/s eta 0:00

# Cell 25 — Load Caspi

In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

CASPI_MODEL_ID = "OzLabs/Caspi-1.7B"

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

caspi_model = Qwen3ASRModel.from_pretrained(
    CASPI_MODEL_ID,
    dtype=dtype,
    device_map="cuda:0",
    max_inference_batch_size=4,
    max_new_tokens=256,
)

print("Caspi loaded.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.08G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Caspi loaded.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
print("pandas:", pd.__version__)

from qwen_asr import Qwen3ASRModel
print("qwen_asr imported successfully")

pandas: 2.3.3
qwen_asr imported successfully


In [ ]:
import os, glob

CHUNKS_DIR = "/content/facebook_test/chunks_30s"

chunk_paths = sorted(glob.glob(f"{CHUNKS_DIR}/chunk_*.wav"))

chunk_files = []

for i, path in enumerate(chunk_paths):
    chunk_files.append({
        "chunk": i,
        "chunk_id": i + 1,
        "start_sec": i * 30,
        "end_sec": (i + 1) * 30,
        "path": path
    })

print("Number of chunks:", len(chunk_files))
print(chunk_files[:3])

Number of chunks: 30
[{'chunk': 0, 'chunk_id': 1, 'start_sec': 0, 'end_sec': 30, 'path': '/content/facebook_test/chunks_30s/chunk_000.wav'}, {'chunk': 1, 'chunk_id': 2, 'start_sec': 30, 'end_sec': 60, 'path': '/content/facebook_test/chunks_30s/chunk_001.wav'}, {'chunk': 2, 'chunk_id': 3, 'start_sec': 60, 'end_sec': 90, 'path': '/content/facebook_test/chunks_30s/chunk_002.wav'}]


In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

CASPI_MODEL_ID = "OzLabs/Caspi-1.7B"

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

caspi_model = Qwen3ASRModel.from_pretrained(
    CASPI_MODEL_ID,
    dtype=dtype,
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    max_inference_batch_size=4,
    max_new_tokens=256,
)

print("Caspi loaded.")

Caspi loaded.


# Cell 26 — Test one chunk

In [ ]:
test_path = chunk_files[0]["path"]

result = caspi_model.transcribe(
    audio=test_path,
    language=None
)

print(result[0].language)
print(result[0].text)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Hebrew
רשת עושים היסטוריה רשת עושים היסטוריה רשת עושים היסטוריה עושים חשבון עם פרופסור עומר מואב בשירה הדס נקר


# Cell 27 — Run Caspi on all chunks

In [ ]:
import pandas as pd
from tqdm import tqdm

rows = []

for item in tqdm(chunk_files):
    try:
        result = caspi_model.transcribe(
            audio=item["path"],
            language=None   # IMPORTANT: do not use "Hebrew"
        )

        text = result[0].text

        try:
            detected_language = result[0].language
        except:
            detected_language = ""

    except Exception as e:
        text = "ERROR: " + str(e)
        detected_language = ""

    rows.append({
        "chunk_id": item["chunk_id"],
        "start_sec": item["start_sec"],
        "end_sec": item["end_sec"],
        "caspi_detected_language": detected_language,
        "caspi_hebrew": text
    })

df_caspi = pd.DataFrame(rows)
df_caspi.head()

100%|██████████| 30/30 [02:59<00:00,  6.00s/it]


,chunk_id,start_sec,end_sec,caspi_detected_language,caspi_hebrew
0,1,0,30,Hebrew,רשת עושים היסטוריה רשת עושים היסטוריה רשת עושי...
1,2,30,60,Hebrew,"אשראי לעסקים קטנים, עם הגר פרץ דיין, שלום. שלו..."
2,3,60,90,Hebrew,"חשוב בכלל מבחינת מה קרו, מה זה כאילו למזיזים מ..."
3,4,90,120,Hebrew,תוצר הגדרה ומדידת התוצר והזהויות החשבונאיות. ר...
4,5,120,150,Hebrew,זו זהות חשבונאית. ומה זה חיסכון? בואו נעשה פה ...


# Cell 28 — Compute Caspi WER/CER

In [ ]:
!pip -q install jiwer

import json, re
import pandas as pd
from jiwer import wer, cer

# Path to your Hebrew reference JSON
REF_HEBREW_JSON = "/content/drive/MyDrive/hebrew/hebrew_youtube_transcript/youtube_hebrew_transcript_30s_chunks_first15min.json"

# If your file is directly inside /hebrew, use this instead:
# REF_HEBREW_JSON = "/content/drive/MyDrive/hebrew/youtube_hebrew_transcript_30s_chunks_first15min.json"

with open(REF_HEBREW_JSON, "r", encoding="utf-8") as f:
    ref_data = json.load(f)

# Load chunks from JSON
if isinstance(ref_data, dict) and "chunks" in ref_data:
    df_ref = pd.DataFrame(ref_data["chunks"])
elif isinstance(ref_data, list):
    df_ref = pd.DataFrame(ref_data)
else:
    raise ValueError("Unknown reference JSON structure")

print("Reference columns:", df_ref.columns.tolist())
print("Caspi columns:", df_caspi.columns.tolist())

# Find Hebrew reference column automatically
possible_text_cols = [
    "youtube_hebrew",
    "youtube_hebrew_reference",
    "hebrew_text",
    "text",
    "transcript",
    "reference"
]

ref_text_col = None
for col in possible_text_cols:
    if col in df_ref.columns:
        ref_text_col = col
        break

if ref_text_col is None:
    raise ValueError(f"No Hebrew text column found. Columns are: {df_ref.columns.tolist()}")

# Make sure chunk_id exists
if "chunk_id" not in df_ref.columns:
    if "chunk" in df_ref.columns:
        df_ref["chunk_id"] = df_ref["chunk"] + 1
    else:
        df_ref["chunk_id"] = range(1, len(df_ref) + 1)

df_ref_small = df_ref[["chunk_id", ref_text_col]].copy()
df_ref_small = df_ref_small.rename(columns={
    ref_text_col: "youtube_hebrew_reference"
})

# Merge reference with Caspi output
df_compare_caspi = pd.merge(
    df_ref_small,
    df_caspi[["chunk_id", "caspi_hebrew"]],
    on="chunk_id",
    how="left"
)

def clean_hebrew_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove Hebrew vowels / niqqud
    text = re.sub(r"[\u0591-\u05C7]", "", text)

    # Keep Hebrew letters and spaces only
    text = re.sub(r"[^\u0590-\u05FF\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_eval_caspi = df_compare_caspi.copy()

df_eval_caspi["ref_clean"] = df_eval_caspi["youtube_hebrew_reference"].apply(clean_hebrew_text)
df_eval_caspi["hyp_clean"] = df_eval_caspi["caspi_hebrew"].apply(clean_hebrew_text)

# Remove empty refs and ERROR rows
df_eval_caspi = df_eval_caspi[
    (df_eval_caspi["ref_clean"] != "") &
    (~df_eval_caspi["caspi_hebrew"].astype(str).str.startswith("ERROR"))
].copy()

print("Valid chunks for Caspi:", len(df_eval_caspi))

if len(df_eval_caspi) == 0:
    raise ValueError("No valid Caspi outputs. All chunks are ERROR or empty.")

# Full 15-min WER/CER
df_eval_caspi = df_eval_caspi.sort_values("chunk_id")

full_ref = " ".join(df_eval_caspi["ref_clean"].tolist())
full_hyp = " ".join(df_eval_caspi["hyp_clean"].tolist())

caspi_wer = wer(full_ref, full_hyp)
caspi_cer = cer(full_ref, full_hyp)

print("===== CASPI FULL 15-MIN METRICS =====")
print("Chunks:", len(df_eval_caspi))
print("WER:", caspi_wer)
print("CER:", caspi_cer)

df_compare_caspi.head()

Reference columns: ['chunk_id', 'start', 'start_sec', 'end', 'end_sec', 'youtube_hebrew']
Caspi columns: ['chunk_id', 'start_sec', 'end_sec', 'caspi_detected_language', 'caspi_hebrew']
Valid chunks for Caspi: 30
===== CASPI FULL 15-MIN METRICS =====
Chunks: 30
WER: 0.2642369020501139
CER: 0.20073122882111646


,chunk_id,youtube_hebrew_reference,caspi_hebrew
0,1,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עו...,רשת עושים היסטוריה רשת עושים היסטוריה רשת עושי...
1,2,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשר...,"אשראי לעסקים קטנים, עם הגר פרץ דיין, שלום. שלו..."
2,3,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הל...,"חשוב בכלל מבחינת מה קרו, מה זה כאילו למזיזים מ..."
3,4,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה...,תוצר הגדרה ומדידת התוצר והזהויות החשבונאיות. ר...
4,5,החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה ...,זו זהות חשבונאית. ומה זה חיסכון? בואו נעשה פה ...


In [ ]:
OUT_DIR = "/content/drive/MyDrive/hebrew/model_comparison_results"
os.makedirs(OUT_DIR, exist_ok=True)

df_compare_caspi.to_excel(f"{OUT_DIR}/caspi_outputs_first15min.xlsx", index=False)
df_compare_caspi.to_csv(f"{OUT_DIR}/caspi_outputs_first15min.csv", index=False, encoding="utf-8-sig")

print("Saved Caspi outputs to:", OUT_DIR)

Saved Caspi outputs to: /content/drive/MyDrive/hebrew/model_comparison_results


## B) Test Ivrit.ai tuned Whisper

This one uses faster-whisper. The model card says it is a Whisper large-v2 model fine-tuned by ivrit.ai to improve Hebrew ASR using crowd-sourced labeling.
#Cell 29 — Install faster-whisper


In [ ]:
!pip -q install -U faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 130.1 MB/s eta 0:00:00


# Cell 30 — Load Ivrit model

In [ ]:
from faster_whisper import WhisperModel
import torch

IVRIT_MODEL_ID = "sivan22/faster-whisper-ivrit-ai-whisper-large-v2-tuned"

device_type = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device_type == "cuda" else "int8"

ivrit_model = WhisperModel(
    IVRIT_MODEL_ID,
    device=device_type,
    compute_type=compute_type
)

print("Ivrit.ai tuned Whisper loaded.")

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

Ivrit.ai tuned Whisper loaded.


# Cell 31 — Test one chunk

In [ ]:
test_path = chunk_files[0]["path"]

segments, info = ivrit_model.transcribe(
    test_path,
    language="he",
    beam_size=5
)

text = " ".join([seg.text.strip() for seg in segments])
print(text)

רשת עושים היסטוריה. עושים חשבון עם פרופסור עומר מואב ושירה הדס נקר. אנה נונן, היי שירה, אנחנו עכשיו בפרק שני על.


# Cell 32 — Run Ivrit on all chunks

In [ ]:
rows = []

for item in tqdm(chunk_files):
    try:
        segments, info = ivrit_model.transcribe(
            item["path"],
            language="he",
            beam_size=5
        )

        text = " ".join([seg.text.strip() for seg in segments])

    except Exception as e:
        text = "ERROR: " + str(e)

    rows.append({
        "chunk_id": item["chunk"] + 1,
        "start_sec": item["start_sec"],
        "end_sec": item["end_sec"],
        "ivrit_hebrew": text
    })

df_ivrit = pd.DataFrame(rows)
df_ivrit.head()

100%|██████████| 30/30 [01:30<00:00,  3.01s/it]


,chunk_id,start_sec,end_sec,ivrit_hebrew
0,1,0,30,רשת עושים היסטוריה. עושים חשבון עם פרופסור עומ...
1,2,30,60,אשראי לעסקים קטנים עם אגר פרץ דיין שלום שלום ש...
2,3,60,90,חשוב בכלל מבחינת מקרו? העסקים האלה מזיזים משהו...
3,4,90,120,"תוצר, הגדרה ומדידת התוצר והזהויות החשבונאיות. ..."
4,5,120,150,זו זהות חשבונאית. ומה זה חיסכון? בואי נעשה פה ...


# Cell 33 — Compute Ivrit WER/CER

In [ ]:
!pip -q install jiwer

import json, re, os
import pandas as pd
from jiwer import wer, cer

# Your Hebrew reference JSON
REF_HEBREW_JSON = "/content/drive/MyDrive/hebrew/hebrew_youtube_transcript/youtube_hebrew_transcript_30s_chunks_first15min.json"

# Load reference
with open(REF_HEBREW_JSON, "r", encoding="utf-8") as f:
    ref_data = json.load(f)

if isinstance(ref_data, dict) and "chunks" in ref_data:
    df_ref = pd.DataFrame(ref_data["chunks"])
elif isinstance(ref_data, list):
    df_ref = pd.DataFrame(ref_data)
else:
    raise ValueError("Unknown reference JSON structure")

print("Reference columns:", df_ref.columns.tolist())
print("Ivrit columns:", df_ivrit.columns.tolist())

# Find Hebrew text column
possible_text_cols = [
    "youtube_hebrew",
    "youtube_hebrew_reference",
    "hebrew_text",
    "text",
    "transcript",
    "reference"
]

ref_text_col = None
for col in possible_text_cols:
    if col in df_ref.columns:
        ref_text_col = col
        break

if ref_text_col is None:
    raise ValueError(f"No Hebrew reference text column found. Columns: {df_ref.columns.tolist()}")

# Make sure chunk_id exists
if "chunk_id" not in df_ref.columns:
    if "chunk" in df_ref.columns:
        df_ref["chunk_id"] = df_ref["chunk"] + 1
    else:
        df_ref["chunk_id"] = range(1, len(df_ref) + 1)

df_ref_small = df_ref[["chunk_id", ref_text_col]].copy()
df_ref_small = df_ref_small.rename(columns={
    ref_text_col: "youtube_hebrew_reference"
})

# Merge reference with Ivrit output
df_compare_ivrit = pd.merge(
    df_ref_small,
    df_ivrit[["chunk_id", "ivrit_hebrew"]],
    on="chunk_id",
    how="left"
)

def clean_hebrew_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove Hebrew vowels / niqqud
    text = re.sub(r"[\u0591-\u05C7]", "", text)

    # Keep Hebrew letters and spaces only
    text = re.sub(r"[^\u0590-\u05FF\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_eval_ivrit = df_compare_ivrit.copy()

df_eval_ivrit["ref_clean"] = df_eval_ivrit["youtube_hebrew_reference"].apply(clean_hebrew_text)
df_eval_ivrit["hyp_clean"] = df_eval_ivrit["ivrit_hebrew"].apply(clean_hebrew_text)

# Remove empty refs and ERROR rows
df_eval_ivrit = df_eval_ivrit[
    (df_eval_ivrit["ref_clean"] != "") &
    (~df_eval_ivrit["ivrit_hebrew"].astype(str).str.startswith("ERROR"))
].copy()

print("Valid chunks for Ivrit:", len(df_eval_ivrit))

if len(df_eval_ivrit) == 0:
    raise ValueError("No valid Ivrit outputs. Rerun the Ivrit transcription cell.")

# Full 15-min WER/CER
df_eval_ivrit = df_eval_ivrit.sort_values("chunk_id")

full_ref = " ".join(df_eval_ivrit["ref_clean"].tolist())
full_hyp = " ".join(df_eval_ivrit["hyp_clean"].tolist())

ivrit_wer = wer(full_ref, full_hyp)
ivrit_cer = cer(full_ref, full_hyp)

print("===== IVRIT.AI WHISPER FULL 15-MIN METRICS =====")
print("Chunks:", len(df_eval_ivrit))
print("WER:", ivrit_wer)
print("CER:", ivrit_cer)

df_compare_ivrit.head()

Reference columns: ['chunk_id', 'start', 'start_sec', 'end', 'end_sec', 'youtube_hebrew']
Ivrit columns: ['chunk_id', 'start_sec', 'end_sec', 'ivrit_hebrew']
Valid chunks for Ivrit: 30
===== IVRIT.AI WHISPER FULL 15-MIN METRICS =====
Chunks: 30
WER: 0.2501138952164009
CER: 0.1920813269127876


,chunk_id,youtube_hebrew_reference,ivrit_hebrew
0,1,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עו...,רשת עושים היסטוריה. עושים חשבון עם פרופסור עומ...
1,2,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשר...,אשראי לעסקים קטנים עם אגר פרץ דיין שלום שלום ש...
2,3,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הל...,חשוב בכלל מבחינת מקרו? העסקים האלה מזיזים משהו...
3,4,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה...,"תוצר, הגדרה ומדידת התוצר והזהויות החשבונאיות. ..."
4,5,החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה ...,זו זהות חשבונאית. ומה זה חיסכון? בואי נעשה פה ...


# **C) Optional: Test base Qwen3-ASR**

Use this only as an experiment. Since base Qwen3-ASR is not clearly Hebrew-specialized, I expect Caspi to be more relevant.

# Cell 34 — Load Qwen3-ASR base: smile

In [ ]:
from qwen_asr import Qwen3ASRModel
import torch

QWEN_MODEL_ID = "Qwen/Qwen3-ASR-1.7B"

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

qwen_model = Qwen3ASRModel.from_pretrained(
    QWEN_MODEL_ID,
    dtype=dtype,
    device_map="cuda:0",
    max_inference_batch_size=4,
    max_new_tokens=256,
)

print("Qwen3-ASR loaded.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.22G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Qwen3-ASR loaded.


## Cell 35 — Run Qwen base

In [ ]:
rows = []

for item in tqdm(chunk_files):
    try:
        result = qwen_model.transcribe(
            audio=item["path"],
            language=None   # auto-detect, safer than forcing Hebrew
        )
        text = result[0].text
        detected_lang = result[0].language

    except Exception as e:
        text = "ERROR: " + str(e)
        detected_lang = ""

    rows.append({
        "chunk_id": item["chunk"] + 1,
        "start_sec": item["start_sec"],
        "end_sec": item["end_sec"],
        "qwen_language": detected_lang,
        "qwen_hebrew": text
    })

df_qwen = pd.DataFrame(rows)
df_qwen.head()

100%|██████████| 30/30 [03:22<00:00,  6.76s/it]


,chunk_id,start_sec,end_sec,qwen_language,qwen_hebrew
0,1,0,30,Hebrew,רשת אסי מיסטוריה. אסי מיסטוריה. אסי מיסטוריה. ...
1,2,30,60,Hebrew,"אשראי לא סכימ קטנים. אם agar פרץ דיאן, שalom. ..."
2,3,60,90,Hebrew,"השוק בקלה מבחינת מקרו? assets, כי זה זיזי מה ש..."
3,4,90,120,Hebrew,התוצרה גדרה ומדידת התוצר והazioni החישובניות. ...
4,5,120,150,Hebrew,זה זוהה תחשבונאית. ובמה זה חיסאחון בой נאסה ב顺...


## Cell 36 — Compute Qwen WER/CER

In [ ]:
!pip -q install jiwer

import json, re, os
import pandas as pd
from jiwer import wer, cer

REF_HEBREW_JSON = "/content/drive/MyDrive/hebrew/hebrew_youtube_transcript/youtube_hebrew_transcript_30s_chunks_first15min.json"

# Load reference JSON
with open(REF_HEBREW_JSON, "r", encoding="utf-8") as f:
    ref_data = json.load(f)

if isinstance(ref_data, dict) and "chunks" in ref_data:
    df_ref = pd.DataFrame(ref_data["chunks"])
elif isinstance(ref_data, list):
    df_ref = pd.DataFrame(ref_data)
else:
    raise ValueError("Unknown reference JSON structure")

print("Reference columns:", df_ref.columns.tolist())
print("Qwen columns:", df_qwen.columns.tolist())

# Find Hebrew reference column
possible_text_cols = [
    "youtube_hebrew",
    "youtube_hebrew_reference",
    "hebrew_text",
    "text",
    "transcript",
    "reference"
]

ref_text_col = None
for col in possible_text_cols:
    if col in df_ref.columns:
        ref_text_col = col
        break

if ref_text_col is None:
    raise ValueError(f"No Hebrew text column found. Columns: {df_ref.columns.tolist()}")

# Make sure chunk_id exists
if "chunk_id" not in df_ref.columns:
    if "chunk" in df_ref.columns:
        df_ref["chunk_id"] = df_ref["chunk"] + 1
    else:
        df_ref["chunk_id"] = range(1, len(df_ref) + 1)

df_ref_small = df_ref[["chunk_id", ref_text_col]].copy()
df_ref_small = df_ref_small.rename(columns={
    ref_text_col: "youtube_hebrew_reference"
})

# Merge reference with Qwen output
df_compare_qwen = pd.merge(
    df_ref_small,
    df_qwen[["chunk_id", "qwen_language", "qwen_hebrew"]],
    on="chunk_id",
    how="left"
)

def clean_hebrew_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove Hebrew vowels / niqqud
    text = re.sub(r"[\u0591-\u05C7]", "", text)

    # Keep Hebrew letters and spaces only
    text = re.sub(r"[^\u0590-\u05FF\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_eval_qwen = df_compare_qwen.copy()

df_eval_qwen["ref_clean"] = df_eval_qwen["youtube_hebrew_reference"].apply(clean_hebrew_text)
df_eval_qwen["hyp_clean"] = df_eval_qwen["qwen_hebrew"].apply(clean_hebrew_text)

df_eval_qwen = df_eval_qwen[
    (df_eval_qwen["ref_clean"] != "") &
    (~df_eval_qwen["qwen_hebrew"].astype(str).str.startswith("ERROR"))
].copy()

print("Valid chunks for Qwen:", len(df_eval_qwen))
print("Detected languages:")
print(df_qwen["qwen_language"].value_counts())

if len(df_eval_qwen) == 0:
    raise ValueError("No valid Qwen outputs. Wait for Qwen transcription cell to finish or check errors.")

# Full 15-min WER/CER
df_eval_qwen = df_eval_qwen.sort_values("chunk_id")

full_ref = " ".join(df_eval_qwen["ref_clean"].tolist())
full_hyp = " ".join(df_eval_qwen["hyp_clean"].tolist())

qwen_wer = wer(full_ref, full_hyp)
qwen_cer = cer(full_ref, full_hyp)

print("===== QWEN FULL 15-MIN METRICS =====")
print("Chunks:", len(df_eval_qwen))
print("WER:", qwen_wer)
print("CER:", qwen_cer)

df_compare_qwen.head()

Reference columns: ['chunk_id', 'start', 'start_sec', 'end', 'end_sec', 'youtube_hebrew']
Qwen columns: ['chunk_id', 'start_sec', 'end_sec', 'qwen_language', 'qwen_hebrew']
Valid chunks for Qwen: 30
Detected languages:
qwen_language
Hebrew    30
Name: count, dtype: int64
===== QWEN FULL 15-MIN METRICS =====
Chunks: 30
WER: 0.6847380410022779
CER: 0.4012841091492777


,chunk_id,youtube_hebrew_reference,qwen_language,qwen_hebrew
0,1,רשת עושים היסטוריה. רשת עושים היסטוריה. רשת עו...,Hebrew,רשת אסי מיסטוריה. אסי מיסטוריה. אסי מיסטוריה. ...
1,2,אלה נו. היי שירה. אנחנו עכשיו בפרק שני. על אשר...,Hebrew,"אשראי לא סכימ קטנים. אם agar פרץ דיאן, שalom. ..."
2,3,רגולציה אבל זה חשוב בכלל מבחינת מקרו העסקים הל...,Hebrew,"השוק בקלה מבחינת מקרו? assets, כי זה זיזי מה ש..."
3,4,למעשה סדרה של פרקים על מדידת התוצר כלומר מה זה...,Hebrew,התוצרה גדרה ומדידת התוצר והazioni החישובניות. ...
4,5,החיסכון שווה בדיוק להשקעה. זוות חשבונאית. ומה ...,Hebrew,זה זוהה תחשבונאית. ובמה זה חיסאחון בой נאסה ב顺...


### D) Final summary table

In [ ]:
results_summary2 = pd.DataFrame([
    {
        "model": "Facebook SeamlessM4T v2 large",
        "task": "Hebrew ASR",
        "WER": 0.6542141230068337,
        "CER": 0.5408418048867487
    },
    {
        "model": "Whisper large-v3",
        "task": "Hebrew ASR",
        "WER": 0.271526,
        "CER": 0.219815
    },
    {
        "model": "Caspi-1.7B",
        "task": "Hebrew ASR",
        "WER": caspi_wer,
        "CER": caspi_cer
    },
    {
        "model": "Ivrit.ai Whisper large-v2 tuned",
        "task": "Hebrew ASR",
        "WER": ivrit_wer,
        "CER": ivrit_cer
    },
    {
        "model": "Qwen3-ASR-1.7B base",
        "task": "Hebrew ASR",
        "WER": qwen_wer,
        "CER": qwen_cer
    }
])

results_summary2

,model,task,WER,CER
0,Facebook SeamlessM4T v2 large,Hebrew ASR,0.654214,0.540842
1,Whisper large-v3,Hebrew ASR,0.271526,0.219815
2,Caspi-1.7B,Hebrew ASR,0.264237,0.200731
3,Ivrit.ai Whisper large-v2 tuned,Hebrew ASR,0.250114,0.192081
4,Qwen3-ASR-1.7B base,Hebrew ASR,0.684738,0.401284


# E) Save **everything**

In [ ]:
import os
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/hebrew/model_comparison_results"
os.makedirs(OUT_DIR, exist_ok=True)

# Create summary table manually from available results
rows = [
    {
        "model": "Facebook SeamlessM4T v2 large",
        "task": "Hebrew ASR",
        "WER": 0.6542141230068337,
        "CER": 0.5408418048867487
    },
    {
        "model": "Whisper large-v3",
        "task": "Hebrew ASR",
        "WER": 0.271526,
        "CER": 0.219815
    }
]

if "caspi_wer" in globals():
    rows.append({
        "model": "Caspi-1.7B",
        "task": "Hebrew ASR",
        "WER": caspi_wer,
        "CER": caspi_cer
    })

if "ivrit_wer" in globals():
    rows.append({
        "model": "Ivrit.ai tuned Whisper",
        "task": "Hebrew ASR",
        "WER": ivrit_wer,
        "CER": ivrit_cer
    })

if "qwen_wer" in globals():
    rows.append({
        "model": "Qwen3-ASR base",
        "task": "Hebrew ASR",
        "WER": qwen_wer,
        "CER": qwen_cer
    })

results_summary_final = pd.DataFrame(rows)

# Save summary
results_summary_final.to_excel(
    f"{OUT_DIR}/all_models_summary_first15min.xlsx",
    index=False
)

results_summary_final.to_csv(
    f"{OUT_DIR}/all_models_summary_first15min.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved summary to:", OUT_DIR)
results_summary_final

Saved summary to: /content/drive/MyDrive/hebrew/model_comparison_results


,model,task,WER,CER
0,Facebook SeamlessM4T v2 large,Hebrew ASR,0.654214,0.540842
1,Whisper large-v3,Hebrew ASR,0.271526,0.219815
2,Caspi-1.7B,Hebrew ASR,0.264237,0.200731
3,Ivrit.ai tuned Whisper,Hebrew ASR,0.250114,0.192081
4,Qwen3-ASR base,Hebrew ASR,0.684738,0.401284


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

matches = list(Path("/content/drive/MyDrive").rglob("hebrew_transcript_model_evaluation.ipynb"))

for m in matches:
    print(m)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/hebrew/hebrew_transcript_model_evaluation.ipynb


In [ ]:
import json
from pathlib import Path

notebook_path = Path("/content/drive/MyDrive/hebrew_transcript_model_evaluation.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb.get("metadata", {}).pop("widgets", None)

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Fixed. metadata.widgets removed.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/hebrew_transcript_model_evaluation.ipynb'

In [7]:
import json
from pathlib import Path

notebook_path = Path("/content/drive/MyDrive/hebrew/hebrew_transcript_model_evaluation.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb.get("metadata", {}).pop("widgets", None)

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Fixed. metadata.widgets removed.")

Fixed. metadata.widgets removed.
